# 03 — Modeling — House Price Predictor

**Input:**
- `data/processed/X_train.csv`
- `data/processed/y_train.csv`

**Output:**
- `models/*_final.pkl` — trained models
- `models/*_metadata.json` — params and CV scores
- `data/processed/oof_predictions.csv` — for ensembling
- `data/processed/test_predictions.csv` — for evaluation
- `models/leaderboard.csv`

## Rules
1. X_test is NEVER loaded here — only in notebook 04
2. Every model uses the same CV strategy
3. Every result is logged immediately after running
4. No tuning based on test performance

## Task
Regression — predict log1p(SalePrice)
Primary metric: RMSE (lower is better)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# ── Models ────────────────────────────────────────────────────────────────────
from sklearn.dummy         import DummyRegressor
from sklearn.linear_model  import (Ridge, Lasso, ElasticNet,
                                    HuberRegressor, BayesianRidge)
from sklearn.neighbors     import KNeighborsRegressor
from sklearn.svm           import SVR, LinearSVR
from sklearn.tree          import DecisionTreeRegressor
from sklearn.ensemble      import (RandomForestRegressor,
                                    ExtraTreesRegressor,
                                    GradientBoostingRegressor,
                                    AdaBoostRegressor)
from sklearn.neural_network import MLPRegressor
import xgboost  as xgb
import lightgbm as lgb
import catboost as cb

# ── CV and metrics ────────────────────────────────────────────────────────────
from sklearn.model_selection import (cross_val_score, KFold, learning_curve)
from sklearn.metrics import mean_squared_error, make_scorer
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from config import (PROCESSED_DATA_PATH, MODELS_PATH,
                    FIGURES_PATH, RANDOM_SEED, CV_FOLDS)

print("✅ Imports OK")

In [ ]:
# ── Task configuration ────────────────────────────────────────────────────────
TASK    = 'regression'
METRIC  = 'rmse'
CV_TYPE = 'kfold'

# ── CV strategy ───────────────────────────────────────────────────────────────
cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)

# ── Scorer ────────────────────────────────────────────────────────────────────
def rmse_fn(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

scorer        = make_scorer(rmse_fn, greater_is_better=False)
higher_better = False

def cv_mean(scores): return -scores.mean()
def cv_std(scores):  return scores.std()

print(f"CV:     KFold ({CV_FOLDS} folds)")
print(f"Metric: RMSE (lower is better)")

In [ ]:
X_train = pd.read_csv(PROCESSED_DATA_PATH / 'X_train.csv')
X_test  = pd.read_csv(PROCESSED_DATA_PATH / 'X_test.csv')
y_train = pd.read_csv(PROCESSED_DATA_PATH / 'y_train.csv').squeeze()
y_test  = pd.read_csv(PROCESSED_DATA_PATH / 'y_test.csv').squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train range: [{y_train.min():.3f}, {y_train.max():.3f}]")

assert X_train.isnull().sum().sum() == 0, "Missing in X_train"
assert X_test.isnull().sum().sum()  == 0, "Missing in X_test"
print("✅ Data loaded and verified")

In [ ]:
results = []

def run(name, model):
    scores = cross_val_score(model, X_train, y_train,
                              cv=cv, scoring=scorer, n_jobs=-1)
    mean = cv_mean(scores)
    std  = cv_std(scores)

    results.append({
        'model':  name,
        'mean':   mean,
        'std':    std,
        'scores': scores.tolist(),
        'time':   datetime.now().strftime('%H:%M:%S')
    })

    print(f"  {name:<50} RMSE: {mean:.4f} ± {std:.4f}")
    return scores


def leaderboard():
    df = pd.DataFrame(results)[['model', 'mean', 'std']]
    return df.sort_values('mean', ascending=True).reset_index(drop=True)

In [ ]:
print("── BASELINE ─────────────────────────────────────────────")
run('Dummy — predict mean',   DummyRegressor(strategy='mean'))
run('Dummy — predict median', DummyRegressor(strategy='median'))
print("\nEvery model below must beat these or something is wrong.")

In [ ]:
print("\n── LINEAR MODELS ────────────────────────────────────────")

# Ridge — L2 regularization
for alpha in [0.01, 0.1, 1, 10, 100, 500, 1000]:
    run(f'Ridge (alpha={alpha})',
        Ridge(alpha=alpha))

# Lasso — L1, automatic feature selection
for alpha in [0.0001, 0.001, 0.01, 0.1]:
    run(f'Lasso (alpha={alpha})',
        Lasso(alpha=alpha, max_iter=10000))

# ElasticNet — L1 + L2
for l1 in [0.2, 0.5, 0.8]:
    run(f'ElasticNet (l1={l1})',
        ElasticNet(alpha=0.001, l1_ratio=l1, max_iter=10000))

# HuberRegressor — robust to outliers
run('HuberRegressor (epsilon=1.35)',
    HuberRegressor(epsilon=1.35, max_iter=300))

run('HuberRegressor (epsilon=1.75)',
    HuberRegressor(epsilon=1.75, max_iter=300))

# BayesianRidge
run('BayesianRidge',
    BayesianRidge())

In [ ]:
print("\n── K-NEAREST NEIGHBORS ──────────────────────────────────")

for k in [3, 5, 7, 11, 15, 21]:
    run(f'KNN (k={k}, uniform)',
        KNeighborsRegressor(n_neighbors=k,
                             weights='uniform', n_jobs=-1))
    run(f'KNN (k={k}, distance)',
        KNeighborsRegressor(n_neighbors=k,
                             weights='distance', n_jobs=-1))

In [ ]:
print("\n── SUPPORT VECTOR MACHINES ──────────────────────────────")

for C in [0.1, 1.0, 10.0]:
    run(f'LinearSVR (C={C})',
        LinearSVR(C=C, max_iter=5000))

# RBF only if dataset is small enough
if len(X_train) <= 50000:
    for C in [1.0, 10.0, 100.0]:
        run(f'SVR RBF (C={C})',
            SVR(kernel='rbf', C=C, gamma='scale', epsilon=0.1))

In [ ]:
print("\n── DECISION TREE ────────────────────────────────────────")

for depth in [3, 5, 7, 10, 15, None]:
    run(f'DecisionTree (max_depth={depth})',
        DecisionTreeRegressor(max_depth=depth,
                               min_samples_split=20,
                               min_samples_leaf=10,
                               random_state=RANDOM_SEED))

In [ ]:
print("\n── RANDOM FOREST & EXTRA TREES ──────────────────────────")

run('RandomForest (100 trees)',
    RandomForestRegressor(n_estimators=100, n_jobs=-1,
                           random_state=RANDOM_SEED))

run('RandomForest (300 trees)',
    RandomForestRegressor(n_estimators=300, n_jobs=-1,
                           random_state=RANDOM_SEED))

run('RandomForest (300, max_depth=10)',
    RandomForestRegressor(n_estimators=300, max_depth=10,
                           n_jobs=-1, random_state=RANDOM_SEED))

run('RandomForest (300, max_features=0.5)',
    RandomForestRegressor(n_estimators=300, max_features=0.5,
                           n_jobs=-1, random_state=RANDOM_SEED))

run('ExtraTrees (300 trees)',
    ExtraTreesRegressor(n_estimators=300, n_jobs=-1,
                         random_state=RANDOM_SEED))

run('ExtraTrees (300, max_depth=10)',
    ExtraTreesRegressor(n_estimators=300, max_depth=10,
                         n_jobs=-1, random_state=RANDOM_SEED))

In [ ]:
print("\n── BOOSTING MODELS ──────────────────────────────────────")

# Sklearn GradientBoosting
run('GradientBoosting (default)',
    GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                               max_depth=3, random_state=RANDOM_SEED))

run('GradientBoosting (slow learner)',
    GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                               max_depth=4, subsample=0.8,
                               random_state=RANDOM_SEED))

# XGBoost
run('XGBoost (default)',
    xgb.XGBRegressor(n_estimators=300, learning_rate=0.1,
                      max_depth=6, subsample=1.0,
                      colsample_bytree=1.0,
                      random_state=RANDOM_SEED,
                      verbosity=0, n_jobs=-1))

run('XGBoost (slow learner)',
    xgb.XGBRegressor(n_estimators=1000, learning_rate=0.05,
                      max_depth=5, subsample=0.8,
                      colsample_bytree=0.8,
                      reg_alpha=0.1, reg_lambda=1.0,
                      random_state=RANDOM_SEED,
                      verbosity=0, n_jobs=-1))

# LightGBM
run('LightGBM (default)',
    lgb.LGBMRegressor(n_estimators=300, learning_rate=0.1,
                       num_leaves=31, random_state=RANDOM_SEED,
                       verbose=-1, n_jobs=-1))

run('LightGBM (slow learner)',
    lgb.LGBMRegressor(n_estimators=1000, learning_rate=0.05,
                       num_leaves=63, subsample=0.8,
                       colsample_bytree=0.8,
                       random_state=RANDOM_SEED,
                       verbose=-1, n_jobs=-1))

run('LightGBM (many leaves)',
    lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05,
                       num_leaves=127, max_depth=8,
                       random_state=RANDOM_SEED,
                       verbose=-1, n_jobs=-1))

# CatBoost
run('CatBoost (default)',
    cb.CatBoostRegressor(iterations=300, learning_rate=0.1,
                          depth=6, random_seed=RANDOM_SEED,
                          verbose=False))

run('CatBoost (slow learner)',
    cb.CatBoostRegressor(iterations=1000, learning_rate=0.03,
                          depth=6, l2_leaf_reg=3,
                          random_seed=RANDOM_SEED,
                          verbose=False))

# AdaBoost
run('AdaBoost (default)',
    AdaBoostRegressor(n_estimators=100, learning_rate=0.1,
                       random_state=RANDOM_SEED))

In [ ]:
print("\n── NEURAL NETWORK (MLP) ─────────────────────────────────")

mlp_common = dict(activation='relu', solver='adam',
                   learning_rate='adaptive',
                   learning_rate_init=0.001,
                   max_iter=500, early_stopping=True,
                   validation_fraction=0.1,
                   random_state=RANDOM_SEED)

run('MLP (128-64)',
    MLPRegressor(hidden_layer_sizes=(128, 64),
                  alpha=0.001, **mlp_common))

run('MLP (256-128-64)',
    MLPRegressor(hidden_layer_sizes=(256, 128, 64),
                  alpha=0.001, **mlp_common))

run('MLP (256-128-64, high reg)',
    MLPRegressor(hidden_layer_sizes=(256, 128, 64),
                  alpha=0.01, **mlp_common))

In [ ]:
lb = leaderboard()
print(lb.to_string())

fig, ax = plt.subplots(figsize=(12, max(6, len(lb) * 0.35)))
colors = ['#2ecc71' if i < 3 else '#3498db' for i in range(len(lb))]
ax.barh(lb['model'], lb['mean'], xerr=lb['std'],
         color=colors, alpha=0.85, capsize=3)
ax.set_xlabel('CV RMSE (lower is better)')
ax.set_title('Regression Model Leaderboard — All Defaults')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_PATH / 'modeling_leaderboard_defaults.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5:")
print(lb.head(5).to_string())

In [ ]:

top_model =Ridge(alpha=0.01)

train_sizes, train_scores, val_scores = learning_curve(
    top_model, X_train, y_train,
    cv=cv, scoring=scorer,
    train_sizes=np.linspace(0.1, 1.0, 10),
    n_jobs=-1
)

tr_mean = -train_scores.mean(axis=1)
vl_mean = -val_scores.mean(axis=1)
gap     = abs(tr_mean[-1] - vl_mean[-1])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, tr_mean, 'o-', label='Training RMSE')
ax.plot(train_sizes, vl_mean, 'o-', label='Validation RMSE')
ax.fill_between(train_sizes,
                 tr_mean - train_scores.std(axis=1),
                 tr_mean + train_scores.std(axis=1), alpha=0.15)
ax.fill_between(train_sizes,
                 vl_mean - val_scores.std(axis=1),
                 vl_mean + val_scores.std(axis=1), alpha=0.15)

if gap > 0.03:
    diag = "High variance — try more regularization"
elif vl_mean[-1] > 0.2:
    diag = "High bias — try more features or complex model"
else:
    diag = "Healthy fit ✅"

#ax.text(0.02, 0.97, diag, transform=ax.transAxes,
       #fontsize=10, va='top',
       #bbox=dict(facecolor='lightyellow', alpha=0.8))
#ax.set(xlabel='Training examples', ylabel='RMSE',
       #title=f'Learning Curve — LightGBM  (gap={gap:.4f})')
ax.legend()
plt.tight_layout()
#plt.savefig(FIGURES_PATH / 'learning_curve_best.png',
            #dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def make_objective(model_fn):
    def objective(trial):
        model  = model_fn(trial)
        scores = cross_val_score(model, X_train, y_train,
                                  cv=cv, scoring=scorer, n_jobs=-1)
        return cv_mean(scores)
    return objective


# ── XGBoost ───────────────────────────────────────────────────────────────────
def xgb_fn(trial):
    return xgb.XGBRegressor(
        n_estimators      = trial.suggest_int('n_estimators', 200, 3000),
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        max_depth         = trial.suggest_int('max_depth', 3, 10),
        min_child_weight  = trial.suggest_int('min_child_weight', 1, 10),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.4, 1.0),
        colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        gamma             = trial.suggest_float('gamma', 0.0, 5.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        random_state=RANDOM_SEED, verbosity=0, n_jobs=-1
    )

study_xgb = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_xgb.optimize(make_objective(xgb_fn), n_trials=100, show_progress_bar=True)
print(f"Best XGBoost RMSE: {study_xgb.best_value:.4f}")
print(json.dumps(study_xgb.best_params, indent=2))


# ── LightGBM ──────────────────────────────────────────────────────────────────
def lgb_fn(trial):
    return lgb.LGBMRegressor(
        n_estimators      = trial.suggest_int('n_estimators', 200, 3000),
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        num_leaves        = trial.suggest_int('num_leaves', 20, 300),
        max_depth         = trial.suggest_int('max_depth', 3, 12),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 100),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.4, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        random_state=RANDOM_SEED, verbose=-1, n_jobs=-1
    )

study_lgb = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_lgb.optimize(make_objective(lgb_fn), n_trials=100, show_progress_bar=True)
print(f"Best LightGBM RMSE: {study_lgb.best_value:.4f}")


# ── CatBoost ──────────────────────────────────────────────────────────────────
def cat_fn(trial):
    return cb.CatBoostRegressor(
        iterations        = trial.suggest_int('iterations', 200, 2000),
        learning_rate     = trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        depth             = trial.suggest_int('depth', 4, 10),
        l2_leaf_reg       = trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bylevel = trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        random_seed=RANDOM_SEED, verbose=False
    )

study_cat = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_cat.optimize(make_objective(cat_fn), n_trials=100, show_progress_bar=True)
print(f"Best CatBoost RMSE: {study_cat.best_value:.4f}")


# ── Ridge ─────────────────────────────────────────────────────────────────────
def ridge_fn(trial):
    return Ridge(alpha=trial.suggest_float('alpha', 1e-3, 1e4, log=True))

study_ridge = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_ridge.optimize(make_objective(ridge_fn), n_trials=50, show_progress_bar=True)
print(f"Best Ridge RMSE:    {study_ridge.best_value:.4f}")


# ── RandomForest ──────────────────────────────────────────────────────────────
def rf_fn(trial):
    return RandomForestRegressor(
        n_estimators      = trial.suggest_int('n_estimators', 100, 1000),
        max_depth         = trial.suggest_int('max_depth', 3, 20),
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20),
        min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 10),
        max_features      = trial.suggest_float('max_features', 0.1, 1.0),
        random_state=RANDOM_SEED, n_jobs=-1
    )

study_rf = optuna.create_study(direction='minimize',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_rf.optimize(make_objective(rf_fn), n_trials=50, show_progress_bar=True)
print(f"Best RandomForest RMSE: {study_rf.best_value:.4f}")